In [2]:
import pandas as pd

In [3]:
df = pd.read_parquet("air_quality_complete_dataset.parquet")

In [5]:
print(df.shape)
print(df.columns)
df.head()

(50417, 85)
Index(['Samplingpoint', 'Pollutant', 'Start', 'End', 'Value', 'Unit',
       'AggType', 'Validity', 'Verification', 'ResultTime', 'DataCapture',
       'FkObservationLog', 'station_id', 'pollutant_code', 'source_file',
       'Country', 'B-G Namespace', 'Year', 'Air Quality Network',
       'Air Quality Network Name', 'Timezone', 'Air Quality Station EoI Code',
       'Air Quality Station Nat Code', 'Air Quality Station Name',
       'Sampling Point Id', 'Air Pollutant', 'Longitude', 'Latitude',
       'Altitude', 'Altitude Unit', 'Air Quality Station Area',
       'Air Quality Station Type', 'Operational Activity Begin',
       'Operational Activity End', 'Sample Id', 'Inlet Height',
       'Inlet Height Unit', 'Building Distance', 'Building Distance Unit',
       'Kerb Distance', 'Kerb Distance Unit', 'Distance Source',
       'Distance Source Unit', 'Main Emission Sources', 'Heating Emissions',
       'Heating Emissions Unit', 'Mobile', 'Traffic Emissions',
       'Traff

,Samplingpoint,Pollutant,Start,End,Value,Unit,AggType,Validity,Verification,ResultTime,...,Detection Limit,Detection Limit Unit,Documentation,QA Report,Duration,Duration Unit,Cadence,Cadence Unit,Source Data URL,Imported
0,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-01-30,2025-01-31,29.000000000000000000,ug.m-3,day,1,2,2025-01-31 09:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21
1,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-01-31,2025-02-01,27.000000000000000000,ug.m-3,day,1,2,2025-02-03 09:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21
2,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-02-01,2025-02-02,5.000000000000000000,ug.m-3,day,1,2,2025-02-03 09:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21
3,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-02-02,2025-02-03,10.000000000000000000,ug.m-3,day,1,2,2025-02-03 09:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21
4,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-02-03,2025-02-04,26.000000000000000000,ug.m-3,day,1,3,2025-02-04 08:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21


In [7]:
cols_to_keep = [
    "station_id",
    "Pollutant",
    "Start",
    "Value",
    "Unit",
    "Country",
    "Municipality",
    "Longitude",
    "Latitude",
    "Altitude",
    "Validity",
    "Verification"
]

df = df[cols_to_keep]

In [9]:
# target
df = df.dropna(subset=["Value"])

# tempo
df["Start"] = pd.to_datetime(df["Start"])

# rename per chiarezza
df = df.rename(columns={"Value": "y"})

In [12]:
df["month"] = df["Start"].dt.month
df["day_of_week"] = df["Start"].dt.dayofweek

In [33]:
df["station_idx"] = df["station_id"].astype("category").cat.codes
df = df[(df["Validity"] == 1) & (df["Verification"] >= 2)]
df = df.copy()

In [22]:
print(df["station_id"].nunique())
print(df.groupby("station_id").size().describe())

119
count    119.000000
mean     265.092437
std       84.384511
min        5.000000
25%      275.000000
50%      299.000000
75%      314.000000
max      332.000000
dtype: float64


In [34]:
counts = df.groupby("station_id").size()
valid_stations = counts[counts >= 30].index
df = df[df["station_id"].isin(valid_stations)]

In [31]:
df.groupby("station_id").size().describe()

count    116.000000
mean     271.431034
std       75.471760
min       69.000000
25%      278.500000
50%      300.000000
75%      314.250000
max      332.000000
dtype: float64

In [35]:
# variables for the model
df["Start"] = pd.to_datetime(df["Start"])
df = df.rename(columns={"Value": "y"})
df["station_idx"] = df["station_id"].astype("category").cat.codes
df["month"] = df["Start"].dt.month
df["day_of_week"] = df["Start"].dt.dayofweek